In [1]:
import mysql.connector

In [7]:
pip install mysql-connector-python



   ---------------------------------------- 0.0/16.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/16.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/16.1 MB ? eta -:--:--
   ---------------------------------------- 0.1/16.1 MB 1.7 MB/s eta 0:00:10
    --------------------------------------- 0.2/16.1 MB 3.1 MB/s eta 0:00:06
   - -------------------------------------- 0.7/16.1 MB 5.2 MB/s eta 0:00:03
   -- ------------------------------------- 1.0/16.1 MB 6.3 MB/s eta 0:00:03
   --- ------------------------------------ 1.4/16.1 MB 7.3 MB/s eta 0:00:03
   ---- ----------------------------------- 2.0/16.1 MB 7.9 MB/s eta 0:00:02
   ------ --------------------------------- 2.6/16.1 MB 8.3 MB/s eta 0:00:02
   ------- -------------------------------- 3.2/16.1 MB 9.1 MB/s eta 0:00:02
   --------- ------------------------------ 3.6/16.1 MB 9.0 MB/s eta 0:00:02
   ---------- ----------------------------- 4.1/16.1 MB 9.1 MB/s eta 0:00:02
   ----------- ----


[notice] A new release of pip is available: 24.0 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
conn = mysql.connector.connect(
    host="localhost",  
    user="root",
    password="1234",
    database="warehouse"
)

cursor = conn.cursor()


In [ ]:
API_URL = "http://127.0.0.1:8000/api/barcodes/"  

In [ ]:
# def send_to_backend(data):
#     payload = {
         
#     }

In [3]:
# Import necessary libraries
import cv2  # OpenCV for image processing
import csv  # CSV module for saving barcode data
from pyzbar.pyzbar import decode  # Pyzbar for decoding barcodes

# Barcode Detector Class

# If a barcode is detected, we need to check the database (identification table) to see whether this barcode ID exists.
class BarcodeDetector:
    def decode_barcodes(self, img, roi):
        barcodes = decode(roi)  # Decode all barcodes in the region of interest
        for barcode in barcodes:
            data = barcode.data.decode('utf-8')  # Extract and decode barcode data
            barcode_type = barcode.type  # Get the barcode type (e.g., QR Code, EAN-13, etc.)
            (x, y, w, h) = barcode.rect  # Get barcode bounding box coordinates

            # Draw a green rectangle around the detected barcode
            cv2.rectangle(img, (x, y), (x + w, y + h), (0, 255, 0), 2)

            # Create a text label with barcode data and type
            text = f"{data} ({barcode_type})"
            cv2.putText(img, text, (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2)

            # check if the barcode exists in the database
            exists = self.update_asset_status(data)
            if exists == True:
                print("found")
                product_info = self.get_product_info(data)
            # Display product details if it exists
            if product_info:
                print(f"Product Found: {product_info}")  
            else:
                print(f"Barcode {data} NOT found in the database.")

    # This function checks if the information of the product exists. If it does then it will output that info
    def update_asset_status(self,barcode_id):
        query = "SELECT * FROM identification WHERE IdentifierValue = %s"
        execution = cursor.execute(query, (barcode_id,))
        resultset = cursor.fetchall()
        for row in resultset:
            # We need only asset id to get the details of the asset
            asset_id = row[1]  
            print("AssetID:", asset_id)
        if resultset:
            assetFound = True
            query = "UPDATE asset SET AssetFound = 1 WHERE AssetID = %s"
            execution = cursor.execute(query, (asset_id,))            
            conn.commit()
        else: 
            assetFound = False
        return assetFound

    def get_product_info(self, barcode_id):
        # First we select the Asset ID from the table
        query = "SELECT * FROM identification WHERE IdentifierValue = %s"
        execution = cursor.execute(query, (barcode_id,))
        resultset = cursor.fetchall()
        for row in resultset:
            # We need only asset id to get the details of the asset
            asset_id = row[1]  
            print("AssetID:", asset_id)
            
        # This provides the data of the asset 
        query = "SELECT * FROM asset WHERE AssetID = %s"
        execution = cursor.execute(query, (asset_id,))
        resultset = cursor.fetchone()  # Fetch the product info if found
        product_id = resultset[1]
        assetLocation = resultset[2]
        assetCondition = resultset[3]
        dateadded = resultset[4]
        assetDispatched = resultset[6]
        print(assetLocation)

        # Getting additional information from the product tabls
        query = "SELECT * FROM product WHERE ProductID = %s"
        execution = cursor.execute(query, (product_id,))
        resultset = cursor.fetchall()
        for row in resultset:
            productName = row[2]
            productCategory = row[3]
            productQuantity = row[5]
            productPrice = row[6]
            print(productName)
            print(productPrice)
        # conn.close()  # Close database connection
        return resultset  # Returns product details (e.g., name, price) or None
        
# Initialize video capture (Webcam)
cap = cv2.VideoCapture(0)  # Open the default camera (index 0)

# Set the resolution of the video capture
cap.set(3, 720)  # Set width to 720 pixels
cap.set(4, 720)  # Set height to 720 pixels


True

In [4]:
import cv2
import csv
import math
from ultralytics import YOLO

class QRCodeDetector:
    
    def decode_qrcodes(self, img, roi):
        detector = cv2.QRCodeDetector()
        data, bbox, _ = detector.detectAndDecode(roi)


        print(f"Decoded Data: {data}")  # Debugging output
        if bbox is not None:
            print("Bounding box detected for QR code.")
        else:
            print("No bounding box detected.")
        
        product_info = None  

        exists = self.update_asset_status(data)
        if exists == True:
            print("found")
            product_info = self.get_product_info(data)
        # Display product details if it exists
        if product_info:
            print(f"Product Found: {product_info}")  
        else:
            print(f"QR Code {data} NOT found in the database.")

        if bbox is not None:
            bbox = bbox.astype(int)
            for i in range(len(bbox)):
                start_point = tuple(bbox[i][0])
                end_point = tuple(bbox[(i + 1) % len(bbox)][0])
                cv2.line(img, start_point, end_point, color=(255, 0, 0), thickness=2)
            cv2.putText(img, data, (int(bbox[0][0][0]), int(bbox[0][0][1]) - 10), 
                        cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 250, 120), 2)

    # This function checks if the information of the product exists. If it does then it will output that info
    def update_asset_status(self,barcode_id):
        query = "SELECT * FROM identification WHERE IdentifierValue = %s"
        execution = cursor.execute(query, (barcode_id,))
        resultset = cursor.fetchall()
        for row in resultset:
            # We need only asset id to get the details of the asset
            asset_id = row[1]  
            print("AssetID:", asset_id)
        if resultset:
            assetFound = True
            query = "UPDATE asset SET AssetFound = 1 WHERE AssetID = %s"
            execution = cursor.execute(query, (asset_id,))            
            conn.commit()
        else: 
            assetFound = False
        return assetFound

    def get_product_info(self, barcode_id):
        # First we select the Asset ID from the table
        query = "SELECT * FROM identification WHERE IdentifierValue = %s"
        execution = cursor.execute(query, (barcode_id,))
        resultset = cursor.fetchall()
        for row in resultset:
            # We need only asset id to get the details of the asset
            asset_id = row[1]  
            print("AssetID:", asset_id)
            
        # This provides the data of the asset 
        query = "SELECT * FROM asset WHERE AssetID = %s"
        execution = cursor.execute(query, (asset_id,))
        resultset = cursor.fetchone()  # Fetch the product info if found
        product_id = resultset[1]
        assetLocation = resultset[2]
        assetCondition = resultset[3]
        dateadded = resultset[4]
        assetDispatched = resultset[6]
        print(assetLocation)

        # Getting additional information from the product tabls
        query = "SELECT * FROM product WHERE ProductID = %s"
        execution = cursor.execute(query, (product_id,))
        resultset = cursor.fetchall()
        for row in resultset:
            productName = row[2]
            productCategory = row[3]
            productQuantity = row[5]
            productPrice = row[6]
            print(productName)
            print(productPrice)
        # conn.close()  # Close database connection
        return resultset  # Returns product details (e.g., name, price) or None

In [27]:
import csv
# def initialize_csv():
#     # with open("Database.csv", mode='w') as csvfile:
    #     csvfileWriter = csv.writer(csvfile, delimiter=',', quotechar='"', quoting=csv.QUOTE_ALL)
    #     csvfileWriter.writerow(["Data", "Detection Count"])

def main():
    print("Choose Detection Mode:")
    print("1. QR Code Detection")
    print("2. Barcode Detection")

    # Initialize the BarcodeDetector and QRCodeDetector
    barcode_detector = BarcodeDetector()
    qrcode_detector = QRCodeDetector()
    
    # Load your custom-trained YOLO model
    model = YOLO("best.pt")



    # Define the classes for your custom dataset
    classNames = ['barcode', 'qrcode']  # Replace with your custom class names
    cap = cv2.VideoCapture(0)
    cap.set(3, 720)
    cap.set(4, 720)



    choice = input("Enter 1 or 2: ")

    if choice == "1":
        
        # initialize_csv()  # Optionally reset CSV for fresh runs
        qrcode_detector = QRCodeDetector()

# Capture Loop
        print("Starting detection. Press 'q' to exit.")
        while True:
            success, img = cap.read()  # Read the image from the camera
            if not success:
                break

    # Perform object detection using YOLO
            results = model(img, stream=True)

    # Loop over detected objects
            for r in results:
                boxes = r.boxes  # Get bounding boxes
                for box in boxes:
                    x1, y1, x2, y2 = box.xyxy[0]  # Get bounding box coordinates
                    x1, y1, x2, y2 = map(int, [x1, y1, x2, y2])  # Convert to int

            # Detect the class name of the object
                    cls = int(box.cls[0])
                    classname = classNames[cls]

            # Calculate the confidence of the detected object
                    confidence = math.ceil(box.conf[0] * 100)
                    # print(f'{classname}: {confidence}%')

            # Draw the bounding box on the image
                    color = (0, 255, 0)  # Default color: green
                    if classname == 'barcode':
                        color = (255, 0, 0)  # Blue for barcode
                        # Add barcode processing here if needed

                    elif classname == 'qrcode':
                        color = (0, 0, 255)  # Red for qrcode
                        # Extract the Region of Interest (ROI) for QR code decoding
                        roi = img[y1:y2, x1:x2]
                        padding = 10  # Adjust padding as needed
                        roi = img[max(0, y1 - padding):min(img.shape[0], y2 + padding), max(0, x1 - padding):min(img.shape[1], x2 + padding)]

                
                        qrcode_detector.decode_qrcodes(img, roi)

                    cv2.rectangle(img, (x1, y1), (x2, y2), color, 3)
                # Write the class name and confidence on the image
                    org = (x1, y1 - 10)  # Position the text above the bounding box
                    font = cv2.FONT_HERSHEY_SIMPLEX
                    cv2.putText(img, f"{classname} {confidence:.2f}%", org, font, 0.5, color, 2)

    # Display the processed image
            cv2.imshow("Webcam", img)

    # Exit loop
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break

        cap.release()
        cv2.destroyAllWindows()


    elif choice == "2":
        # initialize_csv()  # Optionally reset CSV for fresh runs
        barcode_detector = BarcodeDetector()
      
        # Capture Loop
        while True:
            success, img = cap.read()  # Read the image from the camera
            if not success:
                break

            results = model(img, stream=True)  # Send the image to YOLO for detection

            # Loop over detected objects
            for r in results:
                boxes = r.boxes  # Get bounding boxes
                for box in boxes:
                    x1, y1, x2, y2 = box.xyxy[0]  # Get bounding box coordinates
                    x1, y1, x2, y2 = map(int, [x1, y1, x2, y2])  # Convert to int

                    # Detect the class name of the object
                    cls = int(box.cls[0])
                    classname = classNames[cls]

                    # Calculate the confidence of the detected object
                    confidence = math.ceil(box.conf[0] * 100)
                    # print(f'{classname}: {confidence}%')

                    # Draw the bounding box on the image
                    color = (0, 255, 0)  # Default color: green
                    if classname == 'barcode':
                        color = (255, 0, 0)  # Blue for barcode

                        # Extract the Region of Interest (ROI) for barcode decoding
                        roi = img[y1:y2, x1:x2]

                        barcode_detector.decode_barcodes(img, roi)

                    elif classname == 'qrcode':
                        color = (0, 0, 255)  # Red for qrcode

                    cv2.rectangle(img, (x1, y1), (x2, y2), color, 3)

                    # Write the class name and confidence on the image
                    org = (x1, y1 - 10)  # Position the text above the bounding box
                    font = cv2.FONT_HERSHEY_SIMPLEX
                    cv2.putText(img, f"{classname} {confidence:.2f}%", org, font, 0.5, color, 2)

            # Create a window to display the image
            cv2.imshow("Webcam", img)

            # Exit loop
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break

        cap.release()
        cv2.destroyAllWindows()

    else:
        print("Invalid choice. Exiting.")


if __name__ == "__main__":
    main()

Choose Detection Mode:
1. QR Code Detection
2. Barcode Detection

0: 480x640 (no detections), 89.0ms
Speed: 2.1ms preprocess, 89.0ms inference, 0.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 157.4ms
Speed: 2.0ms preprocess, 157.4ms inference, 6.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 136.9ms
Speed: 4.4ms preprocess, 136.9ms inference, 0.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 132.0ms
Speed: 0.0ms preprocess, 132.0ms inference, 0.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 137.7ms
Speed: 0.6ms preprocess, 137.7ms inference, 0.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 qrcode, 123.3ms
Speed: 0.0ms preprocess, 123.3ms inference, 0.7ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 120.1ms
Speed: 0.5ms preprocess, 120.1ms inference, 0.0ms postprocess per image at shape (1, 3, 480, 640)